In [2]:
import pandas as pd

articles = pd.read_csv('./articles.csv')
customers = pd.read_csv('./customers.csv')
transactions = pd.read_csv('./transactions_train.csv')

In [3]:
articles['text'] = (
    articles['prod_name'] + ' ' +
    articles['product_type_name'] + ' ' +
    articles['colour_group_name'] + ' ' +
    articles['garment_group_name'] + ' ' +
    articles['detail_desc'].fillna('')
)

In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer('all-MiniLM-L6-v2',device='cuda')
all_texts = articles['text'].tolist()
#all_embeddings = st_model.encode(all_texts, show_progress_bar=True, batch_size=256)
#np.save('article_embeddings.npy', all_embeddings)
all_embeddings = np.load('article_embeddings.npy')

c:\Users\swson23\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [5]:
import faiss
dimension = all_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(all_embeddings.astype('float32'))


######여기까지 임베딩 벡터########

In [6]:
query = "casual summer white t-shirt"
query_embedding = st_model.encode([query])

D, I = index.search(query_embedding.astype('float32'), k=10)

# 중복 제거
seen = set()
results = []
for idx in I[0]:
    prod_name = articles.iloc[idx]['prod_name']
    if prod_name not in seen:
        seen.add(prod_name)
        results.append(idx)
    if len(results) == 5:
        break

print("검색 결과 (중복 제거):")
for i, idx in enumerate(results):
    print(f"{i+1}. {articles.iloc[idx]['prod_name']} - {articles.iloc[idx]['detail_desc']}")

검색 결과 (중복 제거):
1. Summer price tee - T-shirt in soft cotton jersey with a print motif.
2. SG Summer Top - Wide-fitting sports top in fast-drying functional fabric with a print motif on the front, cap sleeves and a longer, sewn-in vest top with a racer back.
3. SUMMER Top - Short, wide, sports top in printed, fast-drying mesh layered over a vest top with a racer back.
4. Summer top - Short, wide, sports top in printed, fast-drying mesh layered over a vest top with a racer back.
5. Summer graphic tee - T-shirt in soft jersey with a motif on the front.


In [7]:
# 구매 많은 유저 한 명 뽑기
top_user = transactions['customer_id'].value_counts().index[0]
user_history = transactions[transactions['customer_id'] == top_user]['article_id'].tolist()

In [8]:
# 유저 히스토리 임베딩 평균
article_id_to_idx = {aid: idx for idx, aid in enumerate(articles['article_id'])}

user_embeddings = []
for aid in user_history:
    if aid in article_id_to_idx:
        idx = article_id_to_idx[aid]
        user_embeddings.append(all_embeddings[idx])

user_vector = np.mean(user_embeddings, axis=0, keepdims=True)

# 추천
D, I = index.search(user_vector.astype('float32'), k=10)

seen = set(user_history)
results = []
for idx in I[0]:
    aid = articles.iloc[idx]['article_id']
    if aid not in seen:
        results.append(idx)
    if len(results) == 5:
        break

print("유저 맞춤 추천:")
for i, idx in enumerate(results):
    print(f"{i+1}. {articles.iloc[idx]['prod_name']} - {articles.iloc[idx]['detail_desc']}")
    
    ##############################

유저 맞춤 추천:
1. LASSIE LINEN L/S - Long-sleeved top in soft linen jersey with a V-neck and rounded hem. Longer at the back.
2. Twiggy - Calf-length dress in mesh with a low-cut back and long sleeves. Seam at the waist, a gently flared skirt and raw edges at the cuffs and hem. Jersey lining.
3. LASSIE LINEN L/S - Long-sleeved top in a linen weave with a V-neck. Slightly longer at the back.
4. LASSIE LINEN L/S - Long-sleeved top in soft linen jersey with a V-neck and rounded hem. Longer at the back.
5. LASSIE LINEN L/S - Long-sleeved top in soft linen jersey with a V-neck and rounded hem. Longer at the back.


In [9]:
seen_names = set()
seen_ids = set(user_history)
results = []
for idx in I[0]:
    aid = articles.iloc[idx]['article_id']
    name = articles.iloc[idx]['prod_name']
    if aid not in seen_ids and name not in seen_names:
        seen_names.add(name)
        results.append(idx)
    if len(results) == 5:
        break

print("유저 맞춤 추천 (중복 제거):")
for i, idx in enumerate(results):
    print(f"{i+1}. {articles.iloc[idx]['prod_name']} - {articles.iloc[idx]['detail_desc']}")

유저 맞춤 추천 (중복 제거):
1. LASSIE LINEN L/S - Long-sleeved top in soft linen jersey with a V-neck and rounded hem. Longer at the back.
2. Twiggy - Calf-length dress in mesh with a low-cut back and long sleeves. Seam at the waist, a gently flared skirt and raw edges at the cuffs and hem. Jersey lining.
3. RAVEN halfzip ls - Fitted running top in fast-drying functional fabric with a stand-up collar and zip at the top. Long sleeves with thumbholes at the cuffs, a small zipped pocket at the back, reflective details and a gently rounded hem. Slightly longer at the back. Soft brushed inside.
4. HEATHER halfzip ls - Fitted running top in fast-drying functional fabric with an elasticated hood and a zip at the top. Long raglan sleeves, a small zipped pocket at the back, reflective details and a gently rounded hem. Slightly longer at the back. Soft brushed inside.
5. ELENA baselayer - Base layer tights in a soft wool blend with an elasticated waist.


In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "./qwen"
tokenizer = AutoTokenizer.from_pretrained(model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [11]:
def rerank_with_llm(query, candidates, top_k=3):
    candidate_text = "\n".join([
        f"{i+1}. {articles.iloc[idx]['prod_name']}: {articles.iloc[idx]['detail_desc']}"
        for i, idx in enumerate(candidates)
    ])
    
    prompt = f"""당신은 패션 추천 전문가입니다. 반드시 한국어로만 답변하세요. 한자나 중국어는 절대 사용하지 마세요.

사용자 검색어: "{query}"

후보 상품:
{candidate_text}

위 후보 중 가장 적합한 {top_k}개를 선택하고 이유를 설명하세요.
형식:
1. [상품 번호] - [이유]
2. [상품 번호] - [이유]
3. [상품 번호] - [이유]"""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(qwen_model.device)
    outputs = qwen_model.generate(**inputs, max_new_tokens=200)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response

In [12]:
query = "나한테 잘 맞는 옷"
query_embedding = st_model.encode([query])
D, I = index.search(query_embedding.astype('float32'), k=10)
result = rerank_with_llm(query, I[0][:5])
print(result)

1. 2 - [Mia l/s top] - 이 옷은 긴 가슴을 자랑하며, 여성스럽고 쾌적한 소재인 토털 섹시 라운드네일과 날카로운 핑크 컬러가 여성미를 더해줍니다. 긴袖는 겨울철에도 활용 가능하며, 기본적인 스타일에 쉽게 매치할 수 있습니다.
2. 3 - [W GARDA SKIRT EQ] - 이 옷은 높은 웨이스트와detachable tie belt가 있어 몸매를 살리는데 효과적이며, 편안하면서도 세련된 디자인이 특징입니다. 티셔츠나 티셔츠 위에 입을 수 있어 다양한 스타일링이 가능합니다.
3. 5 - [Kardashian skirt (


In [13]:
user_vector = np.mean(user_embeddings, axis=0, keepdims=True)

D, I = index.search(user_vector.astype('float32'), k=20)

seen_names = set()
seen_ids = set(user_history)
candidates = []
for idx in I[0]:
    aid = articles.iloc[idx]['article_id']
    name = articles.iloc[idx]['prod_name']
    if aid not in seen_ids and name not in seen_names:
        seen_names.add(name)
        candidates.append(idx)
    if len(candidates) == 5:
        break

result = rerank_with_llm("이 유저의 구매 히스토리 기반 추천", candidates)
print(result)

1. 3 - [Raven halfzip LS] - 이 상품은 사용자의 구매 히스토리에 따라 운동복 히트웨어를 추천하는 것이 적합합니다. 그 이유는 사용자가 이전에 같은 타입의 운동 상품을 구매한 경험이 있기 때문입니다.
2. 4 - [HEATHER halfzip LS] - 이 상품 역시 운동복과 관련된 제품으로, 사용자의 구매 패턴을 고려할 때 적합합니다. 특히 이 상품은 에어링 디테일과 립업 캡슐이 있어 더욱 세부적인 추천 요소가 될 수 있습니다.
3. 1 - [LASSIE LINEN L/S] - 마지막으로, 이 상품은 셔츠와 같은 페미닌한 스타일의 제품으로, 여성 고객의 구


In [14]:

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
import torch
from datasets import load_dataset
dataset = load_dataset("beomi/KoAlpaca-v1.1a")
def format_data(item):
    return {"text": f"### 지시사항:\n{item['instruction']}\n\n### 답변:\n{item['output']}"}

train_dataset = dataset['train'].map(format_data, remove_columns=['instruction', 'output', 'url'])

tokenizer = AutoTokenizer.from_pretrained("./qwen")
tokenizer.pad_token = tokenizer.eos_token

# 토크나이징
def tokenize(item):
    result = tokenizer(
        item["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = train_dataset.map(tokenize, remove_columns=["text"])
print(tokenized_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 21155
})


In [15]:
import torch
from peft import LoraConfig, get_peft_model

qwen_model = AutoModelForCausalLM.from_pretrained(
    "./qwen",
    torch_dtype=torch.float16,
    device_map="cuda"
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

#qwen_model = get_peft_model(qwen_model, lora_config)
#qwen_model.config.use_cache = False
#qwen_model.print_trainable_parameters()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [16]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./qwen_korean",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=100,
    save_steps=500,
    warmup_steps=100,
    report_to="none",
    save_safetensors=False
)

trainer = Trainer(
    model=qwen_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

#trainer.train()

In [17]:
#qwen_model.save_pretrained("./qwen_korean")
#tokenizer.save_pretrained("./qwen_korean")

In [18]:
# 1. 모델 로드
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("./qwen")
base_model = AutoModelForCausalLM.from_pretrained(
    "./qwen",
    torch_dtype=torch.float16,
    device_map="cuda"
)
korean_model = PeftModel.from_pretrained(base_model, "./qwen_korean")
#korean_model.eval()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [19]:
del qwen_model
del base_model
torch.cuda.empty_cache()

In [20]:
def rerank_with_llm(query, candidates, top_k=3):
    candidate_text = "\n".join([
        f"{i+1}. 상품명: {articles.iloc[idx]['prod_name']}, 카테고리: {articles.iloc[idx]['product_type_name']}, 색상: {articles.iloc[idx]['colour_group_name']}, 설명: {articles.iloc[idx]['detail_desc'][:80]}"
        for i, idx in enumerate(candidates)
    ])
    
    prompt = f"""당신은 패션 추천 전문가입니다. 반드시 한국어로만 답변하세요. 한자나 중국어는 절대 사용하지 마세요.

사용자 검색어: "{query}"

후보 상품:
{candidate_text}

위 후보 중 가장 적합한 {top_k}개를 선택하고 이유를 설명하세요.
형식:
[상품 번호] - [이유]
"""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(korean_model.device)
    outputs = korean_model.generate(**inputs, max_new_tokens=500)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response


In [21]:
query = "나한테 잘 맞는 옷"
query_embedding = st_model.encode([query])
D, I = index.search(query_embedding.astype('float32'), k=10)
result = rerank_with_llm(query, I[0][:5])
print(result)

1. [상품명: W GARDA SKIRT EQ, 색상: Beige] - 높은 밸런스와 시원함을 느낄 수 있는 beige 색상의 스커트입니다. 허리가 높아지고, 부드러운 니트와 함께 착용하기 좋습니다.
2. [상품명: W GARDA SKIRT EQ, 색상: Pink] - 이전과 동일한 이유로 선택합니다. 다양한 스타일에 어울리는 색상으로, 여성스럽고 매력적인 이미지를 줍니다.
3. [상품명: Singapore, 색상: Dark Grey] - 디테일이 과한 것이 아닌 단순하면서도 우아한 디자인으로, 다양한 스타일에 어울릴 것입니다. 흰색이나 다른 색상과 함께 착용하기 좋은 편안한 느낌을 줍니다.

이 외에도, 사용자의 키, 몸매, 스타일에 따라 다른 스커트들이 더 적합할 수 있습니다. 하지만 이 세 가지는 대부분의 경우에 적합한 스커트입니다.


In [ ]:
##################################################

In [1]:
from transformers import AutoModel, AutoTokenizer
import torch

embedding_model = AutoModel.from_pretrained(
    "./Qwen3-VL-Embedding-2B",
    torch_dtype=torch.float16,
    device_map="cuda"
)
embedding_tokenizer = AutoTokenizer.from_pretrained("./Qwen3-VL-Embedding-2B")
print("로드 완료!")

`torch_dtype` is deprecated! Use `dtype` instead!
c:\Users\swson23\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
Some weights of Qwen3VLModel were not initialized from the model checkpoint at ./Qwen3-VL-Embedding-2B and are newly initialized: ['language_model.embed_tokens.weight', 'language_model.layers.0.input_layernorm.weight', 'language_model.layers.0.mlp.down_proj.weight', 'language_model.layers.0.mlp.gate_proj.weight', 'language_model.layers.0.mlp.up_proj.weight', 'language_model.layers.0.post_attention_layernorm.weight', 'language_model.layers.0.self_attn.k_norm.weight', 'language_model.layers.0.self_attn.k_proj.weight', 'language_model.layers.0.self_attn.o_proj.weight', 'language_model.layers.0.self_attn.q_norm.weight', 'language_model.layers.0.self_attn.q_proj.weight', 'language_model.layers.0.self_attn.v_proj

로드 완료!


In [38]:
# 모델 클래스 확인
print(type(embedding_model))

# output keys 확인
inputs = embedding_tokenizer("test", return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = embedding_model(**inputs)
print(outputs.keys())

<class 'transformers.models.qwen3_vl.modeling_qwen3_vl.Qwen3VLModel'>
odict_keys(['last_hidden_state', 'rope_deltas'])


In [ ]:
print(f"VRAM 사용량: {torch.cuda.memory_allocated()/1024**3:.2f}GB")
print(f"VRAM 남은 양: {torch.cuda.memory_reserved()/1024**3:.2f}GB")

In [3]:
import chromadb
import pandas as pd
import torch
import torch.nn.functional as F
import numpy as np

# ChromaDB 클라이언트 생성
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="fashion_items")

print("ChromaDB 준비 완료!")

ChromaDB 준비 완료!


In [9]:
def get_embedding(text):
    inputs = embedding_tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        outputs = embedding_model(**inputs)
    embedding = outputs.last_hidden_state[:, -1, :]
    embedding = F.normalize(embedding, p=2, dim=1)
    return embedding.cpu().numpy()

In [8]:
import time

articles = pd.read_csv('./articles.csv')
existing_ids = set(collection.get(limit=200000)['ids'])

def get_embedding_batch(texts):
    inputs = embedding_tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        outputs = embedding_model(**inputs)
    embeddings = outputs.last_hidden_state[:, -1, :]
    embeddings = F.normalize(embeddings, p=2, dim=1)
    return embeddings.cpu().numpy()

batch_size = 32
t = time.time()

for i in range(0, len(articles), batch_size):
    batch = articles.iloc[i:i+batch_size]
    batch = batch[~batch['article_id'].astype(str).isin(existing_ids)]
    if len(batch) == 0:
        continue
    
    texts = [f"{row['prod_name']} {row['product_type_name']} {row['colour_group_name']} {row['detail_desc']}" for _, row in batch.iterrows()]
    embeddings = get_embedding_batch(texts)
    
    collection.add(
        embeddings=embeddings.tolist(),
        documents=texts,
        metadatas=[{"article_id": str(row['article_id']), "prod_name": row['prod_name']} for _, row in batch.iterrows()],
        ids=[str(row['article_id']) for _, row in batch.iterrows()]
    )
    
    if i % 1000 == 0:
        print(f"{i}/{len(articles)} 완료")

print(f"총 시간: {time.time()-t:.2f}초")

4000/105542 완료
8000/105542 완료
12000/105542 완료
16000/105542 완료
20000/105542 완료
24000/105542 완료
28000/105542 완료
32000/105542 완료
36000/105542 완료
40000/105542 완료
44000/105542 완료
48000/105542 완료
52000/105542 완료
56000/105542 완료
60000/105542 완료
64000/105542 완료
68000/105542 완료
72000/105542 완료
76000/105542 완료
80000/105542 완료
84000/105542 완료
88000/105542 완료
92000/105542 완료
96000/105542 완료
100000/105542 완료
104000/105542 완료
총 시간: 557.30초


In [11]:
query = "comfortable summer dress"
query_embedding = get_embedding(query)

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5
)

for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"{i+1}. {meta['prod_name']}")
    print(f"   {doc[:80]}")
    print()

1. Bobo lined trousers
   Bobo lined trousers Trousers Dark Grey Fully lined cargo trousers in a cotton we

2. Beach straight dressed trs
   Beach straight dressed trs Trousers Grey Tailored trousers in woven fabric with 

3. Bagira Thong Mynta 2p
   Bagira Thong Mynta 2p Underwear bottom Pink Lace thong briefs with a mid waist, 

4. Beach straight dressed trs
   Beach straight dressed trs Trousers Dark Orange Tailored trousers in woven fabri

5. Beach straight dressed trs
   Beach straight dressed trs Trousers Black Tailored trousers in woven fabric with



In [33]:
client.delete_collection("fashion_items")
collection = client.get_or_create_collection(name="fashion_items")
print("초기화 완료!")

초기화 완료!


In [28]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained("./Qwen3-VL-Embedding-2B", trust_remote_code=True)
print("processor 로드 완료!")

processor 로드 완료!


In [29]:
import time

sample = articles.head(1000)

def get_embedding_batch(texts):
    batch_messages = [
        [{"role": "user", "content": [{"type": "text", "text": text}]}]
        for text in texts
    ]
    text_inputs = [
        processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        for messages in batch_messages
    ]
    inputs = processor(text=text_inputs, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
    
    with torch.no_grad():
        outputs = embedding_model(**inputs)
    
    embeddings = outputs.last_hidden_state[:, -1, :]
    embeddings = F.normalize(embeddings, p=2, dim=1)
    return embeddings.cpu().numpy()

batch_size = 32
t = time.time()

for i in range(0, len(sample), batch_size):
    batch = sample.iloc[i:i+batch_size]
    texts = [
        f"{row['prod_name']} {row['product_type_name']} {row['colour_group_name']} {row['detail_desc'] if pd.notna(row['detail_desc']) else ''}"
        for _, row in batch.iterrows()
    ]
    embeddings = get_embedding_batch(texts)
    
    collection.add(
        embeddings=embeddings.tolist(),
        documents=texts,
        metadatas=[{"article_id": str(row['article_id']), "prod_name": row['prod_name']} for _, row in batch.iterrows()],
        ids=[str(row['article_id']) for _, row in batch.iterrows()]
    )

print(f"완료: {time.time()-t:.2f}초")
print(f"저장된 아이템 수: {collection.count()}")

완료: 5.27초
저장된 아이템 수: 1000


In [32]:
query = "comfortable summer dress"
query_embedding = get_embedding_batch([query])

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=20
)

for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"{i+1}. {meta['prod_name']}")
    print(f"   {doc[:80]}")
    print()

1. H-string Jersey
   H-string Jersey Hair string Other Yellow Hair elastics.

2. H-string Jersey
   H-string Jersey Hair string Light Grey Hair elastics.

3. H-string Jersey
   H-string Jersey Hair string Black Hair elastics.

4. Magic Gloves 2pack
   Magic Gloves 2pack Gloves Black Fine-knit gloves.

5. Magic Gloves 2pack
   Magic Gloves 2pack Gloves Black Fine-knit gloves.

6. Magic Gloves 2pack
   Magic Gloves 2pack Gloves Black Fine-knit gloves.

7. H-string Jersey
   H-string Jersey Hair string Grey Hair elastics.

8. 10p Basic Terry
   10p Basic Terry Hair string Black Hair elastics with metal clips.

9. H-string Jersey
   H-string Jersey Hair string Dark Green Hair elastics.

10. Magic Gloves 2pack
   Magic Gloves 2pack Gloves Light Orange Fine-knit gloves.

11. Magic Gloves 2pack
   Magic Gloves 2pack Gloves Light Pink Fine-knit gloves.

12. 4p Claw
   4p Claw Hair clip Light Orange Plastic hair claws. Width 2.5 cm.

13. 4p Claw
   4p Claw Hair clip Black Plastic hair claws. W

In [31]:
sample = articles.head(1000)
dress_items = sample[sample['prod_name'].str.contains('dress', case=False, na=False)]
print(f"dress 아이템 수: {len(dress_items)}")
print(dress_items['prod_name'].tolist())

dress 아이템 수: 52
['Knit dress', 'Rihanna dress', 'Alcazar strap dress', 'Alcazar strap dress', 'Alcazar strap dress', 'Alcazar strap dress', 'Alcazar strap dress', 'Alcazar strap dress', 'Alcazar strap dress', 'Alcazar strap dress', 'Alcazar strap dress', 'Alcazar strap dress', 'Alcazar strap dress', 'Alcazar strap dress', 'Alcazar strap dress', 'Aguilera maxidress', 'Aguilera maxidress', 'Aguilera maxidress', 'Aguilera maxidress', 'Aguilera maxidress', 'Aguilera maxidress', 'Aguilera maxidress', 'Aguilera maxidress', 'Aguilera maxidress', 'Basic SS dress', 'Basic SS dress', 'Basic SS dress', 'Basic SS dress', 'Basic SS dress', 'Basic SS dress', 'Basic SS dress', 'Lena Rib Dress (1)', 'Lena Rib Dress (1)', 'Lena Rib Dress (1)', 'Lena Rib Dress (1)', 'Lena Rib Dress (1)', 'Lena Rib Dress (1)', 'Dress LS Basic', 'Dress LS Basic', 'Dress LS Basic', 'Dress LS Basic', 'Dress LS Basic', 'Dress LS Basic', 'LS frill dress', 'Chelsea welldressed trouser', 'Chelsea welldressed trouser', 'Shirtdre

In [26]:
query = "dress"
query_embedding = get_embedding_batch([query])

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5
)

for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"{i+1}. {meta['prod_name']}")

1. Elsa high waist
2. Brandon denim cropped
3. Astaire 1p Overknee
4. Basic 4pk studs
5. Dress LS Basic


In [34]:
def get_embedding_batch(texts, is_query=False):
    if is_query:
        # 검색 쿼리용 prefix
        prefixed = [f"Instruct: Retrieve relevant fashion items\nQuery: {t}" for t in texts]
    else:
        # 문서 인덱싱은 prefix 없이
        prefixed = texts
    
    inputs = embedding_tokenizer(
        prefixed,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to("cuda")
    
    with torch.no_grad():
        outputs = embedding_model(**inputs)
    
    # 실제 마지막 토큰 위치
    attention_mask = inputs['attention_mask']
    last_indices = attention_mask.sum(dim=1) - 1
    batch_size = outputs.last_hidden_state.shape[0]
    embeddings = outputs.last_hidden_state[torch.arange(batch_size), last_indices]
    
    embeddings = F.normalize(embeddings, p=2, dim=1)
    return embeddings.cpu().numpy()

In [35]:
import torch
import torch.nn.functional as F
import pandas as pd

articles = pd.read_csv('./articles.csv')
sample = articles.head(1000)

def get_embedding_batch(texts, is_query=False):
    if is_query:
        prefixed = [f"Instruct: Retrieve relevant fashion items\nQuery: {t}" for t in texts]
    else:
        prefixed = texts

    inputs = embedding_tokenizer(
        prefixed,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        outputs = embedding_model(**inputs)

    attention_mask = inputs['attention_mask']
    last_indices = attention_mask.sum(dim=1) - 1
    batch_size_n = outputs.last_hidden_state.shape[0]
    embeddings = outputs.last_hidden_state[
        torch.arange(batch_size_n), last_indices
    ]

    embeddings = F.normalize(embeddings, p=2, dim=1)
    return embeddings.cpu().numpy()

batch_size = 32

for i in range(0, len(sample), batch_size):
    batch = sample.iloc[i:i+batch_size]
    texts = [
        f"{row['prod_name']} {row['product_type_name']} {row['colour_group_name']} "
        f"{row['detail_desc'] if pd.notna(row['detail_desc']) else ''}"
        for _, row in batch.iterrows()
    ]
    embeddings = get_embedding_batch(texts, is_query=False)

    collection.add(
        embeddings=embeddings.tolist(),
        documents=texts,
        metadatas=[{
            "article_id": str(row['article_id']),
            "prod_name": row['prod_name']
        } for _, row in batch.iterrows()],
        ids=[str(row['article_id']) for _, row in batch.iterrows()]
    )
    if i % 100 == 0:
        print(f"{i}/1000 완료")

print(f"저장된 아이템 수: {collection.count()}")

0/1000 완료
800/1000 완료
저장된 아이템 수: 1000


In [48]:
import sys
sys.path.append("./Qwen3-VL-Embedding-2B")

import torch
import chromadb
import pandas as pd
from scripts.qwen3_vl_embedding import Qwen3VLEmbedder

# 모델 로딩
model = Qwen3VLEmbedder(
    model_name_or_path="./Qwen3-VL-Embedding-2B",
    torch_dtype=torch.bfloat16,
)
print("모델 로딩 완료")

DOC_INSTRUCTION = "Represent the fashion product for retrieval"
QUERY_INSTRUCTION = "Retrieve relevant fashion items"

# ChromaDB 세팅
client = chromadb.PersistentClient(path="./chroma_db")

try:
    client.delete_collection("fashion_items")
    print("기존 collection 삭제")
except:
    pass

collection = client.create_collection(
    name="fashion_items",
    metadata={"hnsw:space": "cosine"}
)

# 인덱싱 (전체)
articles = pd.read_csv('./articles.csv')
print(f"전체 아이템 수: {len(articles)}")
batch_size = 32

for i in range(0, len(articles), batch_size):
    batch = articles.iloc[i:i+batch_size]

    inputs = [
        {
            "text": (
                f"{row['prod_name']} {row['product_type_name']} "
                f"{row['colour_group_name']} "
                f"{row['detail_desc'] if pd.notna(row['detail_desc']) else ''}"
            ),
            "instruction": DOC_INSTRUCTION
        }
        for _, row in batch.iterrows()
    ]

    embeddings = model.process(inputs)

    collection.add(
        embeddings=embeddings.tolist(),
        documents=[inp["text"] for inp in inputs],
        metadatas=[{
            "article_id": str(row['article_id']),
            "prod_name": row['prod_name']
        } for _, row in batch.iterrows()],
        ids=[str(row['article_id']) for _, row in batch.iterrows()]
    )

    if i % 1000 == 0:
        print(f"{i}/{len(articles)} 완료")

print(f"저장된 아이템 수: {collection.count()}")

# 검색
def search(query: str, n_results: int = 10):
    query_emb = model.process([{
        "text": query,
        "instruction": QUERY_INSTRUCTION
    }])

    results = collection.query(
        query_embeddings=query_emb.tolist(),
        n_results=n_results
    )

    docs      = results['documents'][0]
    metadatas = results['metadatas'][0]
    distances = results['distances'][0]

    print(f"\n검색어: '{query}'\n{'─'*50}")
    for rank, (doc, meta, dist) in enumerate(
        zip(docs, metadatas, distances), 1
    ):
        sim = 1 - dist
        print(f"{rank:2}. [sim={sim:.3f}] {meta['prod_name']}")
        print(f"      {doc[:80]}{'...' if len(doc) > 80 else ''}")
    return results

search("dress")
search("blue jacket")

모델 로딩 완료
기존 collection 삭제
전체 아이템 수: 105542
0/105542 완료
4000/105542 완료
8000/105542 완료
12000/105542 완료
16000/105542 완료
20000/105542 완료
24000/105542 완료
28000/105542 완료
32000/105542 완료
36000/105542 완료
40000/105542 완료
44000/105542 완료
48000/105542 완료
52000/105542 완료
56000/105542 완료
60000/105542 완료
64000/105542 완료
68000/105542 완료
72000/105542 완료
76000/105542 완료
80000/105542 완료
84000/105542 완료
88000/105542 완료
92000/105542 완료
96000/105542 완료
100000/105542 완료
104000/105542 완료
저장된 아이템 수: 105542

검색어: 'dress'
──────────────────────────────────────────────────
 1. [sim=0.657] Blueberry dress
      Blueberry dress Dress Dark Blue 
 2. [sim=0.653] Blueberry dress
      Blueberry dress Dress Dark Yellow 
 3. [sim=0.640] Topi dress J
      Topi dress J Dress White 
 4. [sim=0.633] Topi dress J
      Topi dress J Dress Black 
 5. [sim=0.627] Dan
      Dan Dress Black 
 6. [sim=0.626] Topi dress J
      Topi dress J Dress Dark Green 
 7. [sim=0.624] Topi dress J
      Topi dress J Dress Dark Blue 
 8. [s

{'ids': [['542730009',
   '623667004',
   '781613010',
   '654913001',
   '751428001',
   '611768001',
   '781613013',
   '824490003',
   '688704003',
   '662773003']],
 'embeddings': None,
 'documents': [['Christy Pile Jacket (1) Jacket Blue Jacket in soft pile with a high stand-up collar, concealed zip down the front, side pockets and inner ribbing at the cuffs. Lined.',
   'Twisted jacket Jacket Blue Knitted jacket with a stand-up collar, zip down the front, long raglan sleeves and front pockets. Soft, thermal fleece inside.',
   'Jennifer Blazer Dark Blue Long jacket in a soft weave with notch lapels, flap welt front pockets, a gently tapered waist and long sleeves. No fasteners. Lined.',
   'Daniella Jacket Dark Blue Short, straight-cut coat in woven fabric with 3/4-length flounced sleeves, side pockets and no buttons. Unlined.',
   'JACKET BASIC 89 Jacket Blue Jacket in washed, stretch denim with a collar, buttons down the front and buttoned cuffs.',
   'Zlatan V5 Jacket Dark Blu

c:\Users\swson23\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [ ]:
import sys
sys.path.append("./Qwen3-VL-Embedding-2B")

import os
import torch
import chromadb
import pandas as pd
from scripts.qwen3_vl_embedding import Qwen3VLEmbedder

# 모델 로딩
model = Qwen3VLEmbedder(
    model_name_or_path="./Qwen3-VL-Embedding-2B",
    torch_dtype=torch.bfloat16,
)
print("모델 로딩 완료")

INSTRUCTION = "Retrieve fashion items relevant to the query."

# ChromaDB 세팅
client = chromadb.PersistentClient(path="./chroma_db")

try:
    client.delete_collection("fashion_items")
    print("기존 collection 삭제")
except:
    pass

collection = client.create_collection(
    name="fashion_items",
    metadata={"hnsw:space": "cosine"}
)

# 이미지 경로 생성
def get_image_path(article_id):
    folder = str(article_id)[:3]
    return f"./images/{folder}/{article_id}.jpg"

# 인덱싱
articles = pd.read_csv('./articles.csv')
print(f"전체 아이템 수: {len(articles)}")
batch_size = 8

total_with_image = 0
total_without_image = 0

for i in range(0, len(articles), batch_size):
    batch = articles.iloc[i:i+batch_size]

    inputs = []
    for _, row in batch.iterrows():
        text = (
            f"{row['prod_name']} {row['product_type_name']} "
            f"{row['colour_group_name']} "
            f"{row['detail_desc'] if pd.notna(row['detail_desc']) else ''}"
        )
        img_path = get_image_path(row['article_id'])
        inp = {"text": text, "instruction": INSTRUCTION}
        if os.path.exists(img_path):
            inp["image"] = img_path
            total_with_image += 1
        else:
            total_without_image += 1
        inputs.append(inp)

    embeddings = model.process(inputs)

    collection.add(
        embeddings=embeddings.tolist(),
        documents=[inp["text"] for inp in inputs],
        metadatas=[{
            "article_id": str(row['article_id']),
            "prod_name": row['prod_name'],
            "has_image": os.path.exists(get_image_path(row['article_id']))
        } for _, row in batch.iterrows()],
        ids=[str(row['article_id']) for _, row in batch.iterrows()]
    )

    if i % 1000 == 0:
        print(f"{i}/{len(articles)} 완료 | 이미지 있음: {total_with_image} | 없음: {total_without_image}")

print(f"\n저장된 아이템 수: {collection.count()}")
print(f"이미지 있음: {total_with_image} | 없음: {total_without_image}")

# 검색
def search(query: str, n_results: int = 10):
    query_emb = model.process([{
        "text": query,
        "instruction": INSTRUCTION
    }])

    results = collection.query(
        query_embeddings=query_emb.tolist(),
        n_results=n_results
    )

    docs      = results['documents'][0]
    metadatas = results['metadatas'][0]
    distances = results['distances'][0]

    print(f"\n검색어: '{query}'\n{'─'*50}")
    for rank, (doc, meta, dist) in enumerate(
        zip(docs, metadatas, distances), 1
    ):
        sim = 1 - dist
        print(f"{rank:2}. [sim={sim:.3f}] {meta['prod_name']} {'🖼' if meta['has_image'] else '📝'}")
        print(f"      {doc[:80]}{'...' if len(doc) > 80 else ''}")
    return results

search("dress")
search("blue jacket")

SyntaxError: invalid syntax (1251207214.py, line 1)

In [ ]:
import sys
sys.path.append("./Qwen3-VL-Embedding-2B")

import os
import torch
import chromadb
import pandas as pd
from scripts.qwen3_vl_embedding import Qwen3VLEmbedder

# 모델 로딩
model = Qwen3VLEmbedder(
    model_name_or_path="./Qwen3-VL-Embedding-2B",
    torch_dtype=torch.bfloat16,
)
print("모델 로딩 완료")

INSTRUCTION = "Retrieve fashion items relevant to the query."

# ChromaDB 세팅
client = chromadb.PersistentClient(path="./chroma_db")

try:
    client.delete_collection("fashion_items")
    print("기존 collection 삭제")
except:
    pass

collection = client.create_collection(
    name="fashion_items",
    metadata={"hnsw:space": "cosine"}
)

# 이미지 경로 생성
def get_image_path(article_id):
    folder = str(article_id)[:3]
    return f"./images/{folder}/{article_id}.jpg"

# 인덱싱
articles = pd.read_csv('./articles.csv', dtype={'article_id': str})
articles = articles.head(1000)
print(f"전체 아이템 수: {len(articles)}")
batch_size = 8

total_with_image = 0
total_without_image = 0

for i in range(0, len(articles), batch_size):
    batch = articles.iloc[i:i+batch_size]

    inputs = []
    for _, row in batch.iterrows():
        text = (
            f"{row['prod_name']} {row['product_type_name']} "
            f"{row['colour_group_name']} "
            f"{row['detail_desc'] if pd.notna(row['detail_desc']) else ''}"
        )
        img_path = get_image_path(row['article_id'])
        inp = {"text": text, "instruction": INSTRUCTION}
        if os.path.exists(img_path):
            inp["image"] = img_path
            total_with_image += 1
        else:
            total_without_image += 1
        inputs.append(inp)

    embeddings = model.process(inputs)

    collection.add(
        embeddings=embeddings.tolist(),
        documents=[inp["text"] for inp in inputs],
        metadatas=[{
            "article_id": str(row['article_id']),
            "prod_name": row['prod_name'],
            "has_image": os.path.exists(get_image_path(row['article_id']))
        } for _, row in batch.iterrows()],
        ids=[str(row['article_id']) for _, row in batch.iterrows()]
    )

    if i % 100 == 0:
        print(f"{i}/{len(articles)} 완료 | 이미지 있음: {total_with_image} | 없음: {total_without_image}")

print(f"\n저장된 아이템 수: {collection.count()}")
print(f"이미지 있음: {total_with_image} | 없음: {total_without_image}")

# 검색
def search(query: str, n_results: int = 10):
    query_emb = model.process([{
        "text": query,
        "instruction": INSTRUCTION
    }])

    results = collection.query(
        query_embeddings=query_emb.tolist(),
        n_results=n_results
    )

    docs      = results['documents'][0]
    metadatas = results['metadatas'][0]
    distances = results['distances'][0]

    print(f"\n검색어: '{query}'\n{'─'*50}")
    for rank, (doc, meta, dist) in enumerate(
        zip(docs, metadatas, distances), 1
    ):
        sim = 1 - dist
        print(f"{rank:2}. [sim={sim:.3f}] {meta['prod_name']} {'🖼' if meta['has_image'] else '📝'}")
        print(f"      {doc[:80]}{'...' if len(doc) > 80 else ''}")
    return results

search("dress")
search("blue jacket")

모델 로딩 완료
기존 collection 삭제
전체 아이템 수: 1000


In [8]:
import pandas as pd
articles = pd.read_csv('./articles.csv', dtype={'article_id': str})
row = articles.iloc[0]
print(row['article_id'])
print(f"./images/{str(row['article_id'])[:3]}/{row['article_id']}.jpg")

0108775015
./images/010/0108775015.jpg


In [6]:
import sys
sys.path.append("./Qwen3-VL-Embedding-2B")

import os
import torch
import pandas as pd
from scripts.qwen3_vl_embedding import Qwen3VLEmbedder

# 모델 로딩
model = Qwen3VLEmbedder(
    model_name_or_path="./Qwen3-VL-Embedding-2B",
    torch_dtype=torch.bfloat16,
    max_pixels=512*512,
)
print("모델 로딩 완료")

INSTRUCTION = "Retrieve fashion items relevant to the query."

def get_image_path(article_id):
    folder = str(article_id)[:3]
    return f"./images/{folder}/{article_id}.jpg"

# 이미지 하나만 테스트
articles = pd.read_csv('./articles.csv', dtype={'article_id': str})
row = articles.iloc[0]

img_path = get_image_path(row['article_id'])
print(f"이미지 경로: {img_path}")
print(f"파일 존재: {os.path.exists(img_path)}")

test_input = [{
    "image": img_path,
    "text": f"{row['prod_name']} {row['product_type_name']} {row['colour_group_name']}",
    "instruction": INSTRUCTION
}]

emb = model.process(test_input)
print(f"임베딩 shape: {emb.shape}")
print(f"임베딩 첫 5개 값: {emb[0][:5]}")

모델 로딩 완료
이미지 경로: ./images/010/0108775015.jpg
파일 존재: True
임베딩 shape: torch.Size([1, 2048])
임베딩 첫 5개 값: tensor([-0.0177, -0.0062, -0.0103, -0.0110, -0.0022], device='cuda:0',
       dtype=torch.bfloat16)


In [ ]:
articles = pd.read_csv('./articles.csv', dtype={'article_id': str})
sample = articles.head(10)

import time
start = time.time()

for _, row in sample.iterrows():
    img_path = get_image_path(row['article_id'])
    
    test_input = [{
        "image": img_path if os.path.exists(img_path) else None,
        "text": f"{row['prod_name']} {row['product_type_name']} {row['colour_group_name']}",
        "instruction": INSTRUCTION
    }]
    # image None이면 키 제거
    if test_input[0]["image"] is None:
        del test_input[0]["image"]
    
    emb = model.process(test_input)

elapsed = time.time() - start
print(f"100개 완료: {elapsed:.1f}초")
print(f"1개당 평균: {elapsed/100:.2f}초")
print(f"10만개 예상: {elapsed/100*105000/3600:.1f}시간")

100개 완료: 77.6초
1개당 평균: 0.78초
10만개 예상: 22.6시간


In [8]:
import time
start = time.time()

batch = sample.iloc[:8]
inputs = []
for _, row in batch.iterrows():
    img_path = get_image_path(row['article_id'])
    inp = {"text": f"{row['prod_name']} {row['product_type_name']} {row['colour_group_name']}", "instruction": INSTRUCTION}
    if os.path.exists(img_path):
        inp["image"] = img_path
    inputs.append(inp)

emb = model.process(inputs)
elapsed = time.time() - start
print(f"배치 8개: {elapsed:.1f}초")
print(f"10만개 예상: {elapsed/8*105000/3600:.1f}시간")

배치 8개: 60.0초
10만개 예상: 218.6시간


In [10]:
import sys
sys.path.append("./Qwen3-VL-Embedding-2B")

import os
import torch
import chromadb
import pandas as pd
from scripts.qwen3_vl_embedding import Qwen3VLEmbedder

# 모델 로딩
model = Qwen3VLEmbedder(
    model_name_or_path="./Qwen3-VL-Embedding-2B",
    torch_dtype=torch.bfloat16,
    max_pixels=256*256,
)
print("모델 로딩 완료")

INSTRUCTION = "Retrieve fashion items relevant to the query."

# ChromaDB 세팅
client = chromadb.PersistentClient(path="./chroma_db")

try:
    client.delete_collection("fashion_items")
    print("기존 collection 삭제")
except:
    pass

collection = client.create_collection(
    name="fashion_items",
    metadata={"hnsw:space": "cosine"}
)

# 이미지 경로 생성
def get_image_path(article_id):
    folder = str(article_id)[:3]
    return f"./images/{folder}/{article_id}.jpg"

def print_vram():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved  = torch.cuda.memory_reserved() / 1024**3
    print(f"VRAM | 사용중: {allocated:.2f}GB | 예약됨: {reserved:.2f}GB")

# 인덱싱
articles = pd.read_csv('./articles.csv', dtype={'article_id': str})
articles = articles.head(1000)
print(f"전체 아이템 수: {len(articles)}")
batch_size = 16

total_with_image = 0
total_without_image = 0

for i in range(0, len(articles), batch_size):
    batch = articles.iloc[i:i+batch_size]

    inputs = []
    for _, row in batch.iterrows():
        text = (
            f"{row['prod_name']} {row['product_type_name']} "
            f"{row['colour_group_name']} "
            f"{row['detail_desc'] if pd.notna(row['detail_desc']) else ''}"
        )
        img_path = get_image_path(row['article_id'])
        inp = {"text": text, "instruction": INSTRUCTION}
        if os.path.exists(img_path):
            inp["image"] = img_path
            total_with_image += 1
        else:
            total_without_image += 1
        inputs.append(inp)

    embeddings = model.process(inputs)

    collection.add(
        embeddings=embeddings.tolist(),
        documents=[inp["text"] for inp in inputs],
        metadatas=[{
            "article_id": str(row['article_id']),
            "prod_name": row['prod_name'],
            "has_image": os.path.exists(get_image_path(row['article_id']))
        } for _, row in batch.iterrows()],
        ids=[str(row['article_id']) for _, row in batch.iterrows()]
    )

    if i % 100 == 0:
        print(f"{i}/{len(articles)} 완료 | 이미지 있음: {total_with_image} | 없음: {total_without_image}")
        print_vram()

print(f"\n저장된 아이템 수: {collection.count()}")
print(f"이미지 있음: {total_with_image} | 없음: {total_without_image}")
print_vram()

# 검색
def search(query: str, n_results: int = 10):
    query_emb = model.process([{
        "text": query,
        "instruction": INSTRUCTION
    }])

    results = collection.query(
        query_embeddings=query_emb.tolist(),
        n_results=n_results
    )

    docs      = results['documents'][0]
    metadatas = results['metadatas'][0]
    distances = results['distances'][0]

    print(f"\n검색어: '{query}'\n{'─'*50}")
    for rank, (doc, meta, dist) in enumerate(
        zip(docs, metadatas, distances), 1
    ):
        sim = 1 - dist
        print(f"{rank:2}. [sim={sim:.3f}] {meta['prod_name']} {'🖼' if meta['has_image'] else '📝'}")
        print(f"      {doc[:80]}{'...' if len(doc) > 80 else ''}")
    return results

search("dress")
search("blue jacket")

모델 로딩 완료
기존 collection 삭제
전체 아이템 수: 1000
0/1000 완료 | 이미지 있음: 16 | 없음: 0
VRAM | 사용중: 7.94GB | 예약됨: 11.93GB


KeyboardInterrupt: 